# Auto Loan Portfolio Analysis
**Mauricio Morales** | Credit Analyst Portfolio Project

This notebook analyzes a simulated auto loan portfolio of 400 applications (2022–2024).  
It mirrors the type of analysis performed by a credit analyst: approval patterns, risk segmentation, default rates, and portfolio health.

**Tools:** Python · SQLite · Pandas · Matplotlib · Seaborn


## Setup — Load Libraries and Data

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Style
sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

# Connect to database
conn = sqlite3.connect('auto_loan_analysis.db')

# Load both tables
applications = pd.read_sql_query("SELECT * FROM applications", conn)
loan_perf    = pd.read_sql_query("SELECT * FROM loan_performance", conn)

print(f"Applications: {len(applications)} rows")
print(f"Funded Loans: {len(loan_perf)} rows")
applications.head()

## Section 1: Portfolio Overview

In [ ]:
# Overall approval rate
total = len(applications)
approved = (applications['decision'] == 'Approved').sum()
declined = total - approved

print(f"Total Applications : {total}")
print(f"Approved           : {approved} ({approved/total*100:.1f}%)")
print(f"Declined           : {declined} ({declined/total*100:.1f}%)")

# Pie chart
fig, ax = plt.subplots(figsize=(6,5))
ax.pie([approved, declined], labels=['Approved', 'Declined'],
       autopct='%1.1f%%', colors=['#2563eb','#e5e7eb'],
       startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Overall Loan Approval Rate')
plt.tight_layout()
plt.show()

## Section 2: Approval Rate by Credit Tier

In [ ]:
tier_order = ['Super Prime','Prime','Near Prime','Subprime','Deep Subprime']

tier_stats = (applications.groupby('credit_tier')
    .apply(lambda x: pd.Series({
        'total': len(x),
        'approved': (x['decision']=='Approved').sum(),
        'approval_pct': (x['decision']=='Approved').mean() * 100,
        'avg_score': x['credit_score'].mean(),
        'avg_rate': x['interest_rate'].mean()
    }))
    .reindex(tier_order)
    .reset_index())

display(tier_stats.round(2))

fig, ax = plt.subplots()
bars = ax.bar(tier_stats['credit_tier'], tier_stats['approval_pct'], color='#2563eb', edgecolor='white')
ax.set_title('Approval Rate by Credit Tier')
ax.set_ylabel('Approval Rate (%)')
ax.set_ylim(0, 100)
for bar, val in zip(bars, tier_stats['approval_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Section 3: Interest Rate by Credit Tier

In [ ]:
approved_df = applications[applications['decision'] == 'Approved'].copy()

fig, ax = plt.subplots()
sns.boxplot(data=approved_df, x='credit_tier', y='interest_rate', order=tier_order, ax=ax, palette='Blues')
ax.set_title('Interest Rate Distribution by Credit Tier')
ax.set_ylabel('Interest Rate (%)')
ax.set_xlabel('')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Section 4: Default Rate by Credit Tier

In [ ]:
# Merge applications with loan performance
merged = loan_perf.merge(applications[['application_id','credit_tier','credit_score',
                                        'ltv_ratio','dti_ratio','loan_amount','interest_rate']],
                          on='application_id')

merged['is_default'] = (merged['status'] == 'Default').astype(int)

default_by_tier = (merged.groupby('credit_tier')
    .agg(funded_loans=('loan_id','count'),
         defaults=('is_default','sum'),
         default_rate=('is_default','mean'))
    .reindex(tier_order)
    .reset_index())
default_by_tier['default_rate_pct'] = default_by_tier['default_rate'] * 100

display(default_by_tier[['credit_tier','funded_loans','defaults','default_rate_pct']].round(2))

fig, ax = plt.subplots()
colors = ['#22c55e','#86efac','#fbbf24','#f97316','#ef4444']
bars = ax.bar(default_by_tier['credit_tier'], default_by_tier['default_rate_pct'],
              color=colors, edgecolor='white')
ax.set_title('Default Rate by Credit Tier')
ax.set_ylabel('Default Rate (%)')
for bar, val in zip(bars, default_by_tier['default_rate_pct']):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Section 5: Portfolio Health — Loan Status Breakdown

In [ ]:
status_summary = (loan_perf.groupby('status')
    .agg(count=('loan_id','count'),
         total_balance=('remaining_balance','sum'))
    .sort_values('count', ascending=False)
    .reset_index())
status_summary['pct'] = status_summary['count'] / status_summary['count'].sum() * 100

display(status_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count
axes[0].barh(status_summary['status'], status_summary['count'], color='#2563eb')
axes[0].set_title('Loan Count by Status')
axes[0].set_xlabel('Number of Loans')

# Balance exposure
axes[1].barh(status_summary['status'], status_summary['total_balance'], color='#7c3aed')
axes[1].set_title('Balance Exposure by Status ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_xlabel('Remaining Balance')

plt.tight_layout()
plt.show()

## Section 6: LTV Risk Analysis

In [ ]:
# LTV bucket analysis
merged['ltv_bucket'] = pd.cut(merged['ltv_ratio'],
    bins=[0, 0.80, 0.90, 1.00, 1.10, 99],
    labels=['<80%','80-89%','90-99%','100-109%','110%+'])

ltv_risk = (merged.groupby('ltv_bucket', observed=True)
    .agg(loans=('loan_id','count'),
         defaults=('is_default','sum'),
         default_rate=('is_default','mean'))
    .reset_index())
ltv_risk['default_rate_pct'] = ltv_risk['default_rate'] * 100
display(ltv_risk[['ltv_bucket','loans','defaults','default_rate_pct']].round(2))

fig, ax = plt.subplots()
ax.bar(ltv_risk['ltv_bucket'].astype(str), ltv_risk['default_rate_pct'], color='#f97316')
ax.set_title('Default Rate by LTV Bucket')
ax.set_ylabel('Default Rate (%)')
ax.set_xlabel('LTV Range')
plt.tight_layout()
plt.show()

## Section 7: Monthly Application Volume Trend

In [ ]:
applications['application_date'] = pd.to_datetime(applications['application_date'])
applications['month'] = applications['application_date'].dt.to_period('M')

monthly = (applications.groupby(['month','decision'])
    .size().unstack(fill_value=0).reset_index())
monthly['month_str'] = monthly['month'].astype(str)

fig, ax = plt.subplots(figsize=(12,5))
ax.stackplot(monthly['month_str'],
             monthly.get('Approved', 0), monthly.get('Declined', 0),
             labels=['Approved','Declined'], colors=['#2563eb','#e5e7eb'], alpha=0.85)
ax.set_title('Monthly Application Volume — Approved vs Declined')
ax.set_ylabel('Number of Applications')
ax.legend(loc='upper left')
plt.xticks(rotation=45, ha='right')
ax.set_xticks(ax.get_xticks()[::2])
plt.tight_layout()
plt.show()

## Section 8: Credit Score Distribution

In [ ]:
fig, ax = plt.subplots()
ax.hist(applications[applications['decision']=='Approved']['credit_score'],
        bins=20, alpha=0.7, color='#2563eb', label='Approved')
ax.hist(applications[applications['decision']=='Declined']['credit_score'],
        bins=20, alpha=0.7, color='#ef4444', label='Declined')
ax.set_title('Credit Score Distribution — Approved vs Declined')
ax.set_xlabel('Credit Score')
ax.set_ylabel('Number of Applicants')
ax.legend()
plt.tight_layout()
plt.show()

## Section 9: Geographic Distribution

In [ ]:
geo = (applications[applications['decision']=='Approved']
    .groupby('state')
    .agg(funded_loans=('application_id','count'),
         total_funded=('loan_amount','sum'))
    .sort_values('total_funded', ascending=True)
    .reset_index())

fig, ax = plt.subplots(figsize=(8,6))
bars = ax.barh(geo['state'], geo['total_funded'], color='#2563eb')
ax.set_title('Total Funded Loan Volume by State')
ax.set_xlabel('Total Loan Volume ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

## Summary — Key Findings

| Metric | Value |
|---|---|
| Total Applications | 400 |
| Approval Rate | ~48% |
| Funded Loans | 191 |
| Default Rate | ~6% |
| Super Prime Approval | 87% |
| Deep Subprime Approval | 16% |
| Highest Default Tier | Subprime |
| Portfolio Balance Exposure | ~$2.5M (current) |

**Insights:**
- Credit score is the strongest predictor of both approval and default risk
- Subprime tier shows the highest default rate despite moderate approval rates
- LTV ratio above 110% correlates with elevated default risk
- Deep Subprime accounts for 35% of applications but only 11% of funded loans
- Geographic concentration is relatively even across 10 states

*Project by Mauricio Morales — memora5905.github.io*
